# WavLM Evaluation - Embedding-Based Classification (Machine Learning)

This notebook evaluates the predictive power of WavLM embeddings for speaker traits using various machine learning models.

**Validation Strategy**: StratifiedGroupKFold (n=10) by `speaker_id` to ensure health status balance and prevent data leakage.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from IPython.display import display
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Pandas display settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Add root directory to sys.path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from embeddings_eval.data_loader import load_embeddings, load_metadata
from embeddings_eval.analyzer import WavLMAnalyzer
from embeddings_eval.reporter import WavLMReporter
from embeddings_eval.constants import GROUP_DDK

In [ ]:
DATA_DIR = "../datalocal/PC-GITA_v260210_24kHz/speaker_embeddings/wavLM"
META_PATH = "../datalocal/PC-GITA_v260210_24kHz/_metadata/PCGITAtoPD_mapping.csv"

print("Loading data and metadata...")
metadata = load_metadata(META_PATH)
embeddings = load_embeddings(DATA_DIR, metadata=metadata)
analyzer = WavLMAnalyzer(embeddings)

print(f"Loaded {len(embeddings)} samples.")

In [ ]:
# Data preparation for scikit-learn
X = np.stack([e.vector.cpu().numpy() for e in embeddings])
y_sex = np.array([e.sex for e in embeddings])
y_age = np.array([e.age for e in embeddings])
y_status = np.array([e.health_status for e in embeddings])
y_hy = np.array([e.hy for e in embeddings])
groups = np.array([e.speaker_id for e in embeddings])
task_groups = np.array([e.group for e in embeddings])

cv = StratifiedGroupKFold(n_splits=10)

def run_ml_experiment(X, y, groups, model_name='lr', task='clf', label='Experiment', return_proba=False):
    if task == 'clf':
        preds = np.zeros_like(y, dtype=object)
        probas = None
        if return_proba: probas = np.zeros((len(y), 2))
    else:
        preds = np.zeros_like(y, dtype=float)
    
    folds = list(cv.split(X, y, groups))
    for train_idx, test_idx in tqdm(folds, desc=f"Training {label} ({model_name.upper()})"):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        if task == 'clf':
            if model_name == 'lr': model = LogisticRegression(max_iter=1000)
            elif model_name == 'mlp': model = MLPClassifier(hidden_layer_sizes=(256, 128, 64), max_iter=500)
            elif model_name == 'hgbt': model = HistGradientBoostingClassifier()
            
            model.fit(X_train, y_train)
            preds[test_idx] = model.predict(X_test)
            if return_proba: probas[test_idx] = model.predict_proba(X_test)
        else: # Regression
            if model_name == 'ridge': model = Ridge()
            elif model_name == 'hgbt': model = HistGradientBoostingRegressor()
            model.fit(X_train, y_train)
            preds[test_idx] = model.predict(X_test)
        
    if return_proba: return preds, probas, model.classes_
    return preds

## 1. Sex Classification (M vs F)

In [ ]:
for m in ['lr', 'mlp', 'hgbt']:
    p = run_ml_experiment(X, y_sex, groups, model_name=m, task='clf', label='Sex')
    acc = accuracy_score(y_sex, p)
    print(f"Model {m.upper()} Accuracy: {acc*100:.2f}%")

## 2. Age Prediction (Years)

In [ ]:
for m in ['ridge', 'hgbt']:
    p = run_ml_experiment(X, y_age, groups, model_name=m, task='reg', label='Age')
    err = np.abs(y_age - p)
    print(f"Model {m.upper()} MAE: {err.mean():.2f} years (Var: {err.var():.2f})")

## 3. PD vs HC Detection Experiments

**NOTE**: DDK task is excluded to avoid bias. We perform experiments on all data, per task group, and a combined monologue+sentence subset.

In [ ]:
mask_no_ddk = (task_groups != GROUP_DDK)
X_f, y_s_f, gr_f, tg_f = X[mask_no_ddk], y_status[mask_no_ddk], groups[mask_no_ddk], task_groups[mask_no_ddk]

exp_results = {}
MODELS = ['lr', 'mlp', 'hgbt']
SUBSETS = ['All (excl. DDK)', 'monologue', 'readtext', 'sentence', 'words', 'monologue+sentence']

for s_name in SUBSETS:
    if s_name == 'All (excl. DDK)':
        X_curr, y_curr, gr_curr = X_f, y_s_f, gr_f
    elif s_name == 'monologue+sentence':
        mask = (task_groups == 'monologue') | (task_groups == 'sentence')
        X_curr, y_curr, gr_curr = X[mask], y_status[mask], groups[mask]
    else:
        mask = (task_groups == s_name)
        X_curr, y_curr, gr_curr = X[mask], y_status[mask], groups[mask]
    
    print(f"\n>>> Running PD/HC Experiment on subset: {s_name.upper()} <<<")
    for m_name in MODELS:
        p, prob, cl = run_ml_experiment(X_curr, y_curr, gr_curr, model_name=m_name, task='clf', label=f'PD/HC-{s_name}', return_proba=True)
        exp_results[(s_name, m_name)] = (p, prob, cl, y_curr, gr_curr)
        print(f"Model {m_name.upper()} Sample Accuracy: {accuracy_score(y_curr, p)*100:.2f}%")

## 4. Final Summary Evaluation

This table summarizes performance across all subsets and models using three metrics:
1. **Sample-level Acc**: Accuracy calculated over individual files.
2. **Maj. Vote Acc (Conf)**: Speaker-level accuracy (most frequent label). Confidence is the average % of files matching the winner.
3. **Avg. Prob Acc (Prob)**: Speaker-level accuracy (average probability). Probability is the mean confidence score of the predicted class.

In [ ]:
summary_rows = []
for s_name in SUBSETS:
    row = {'Data Subset': s_name}
    for m_name in MODELS:
        p, prob, classes, y_true, s_groups = exp_results[(s_name, m_name)]
        
        # 1. Sample level
        s_acc = accuracy_score(y_true, p) * 100
        
        # 2. Speaker level metrics
        speaker_data = pd.DataFrame({'sid': s_groups, 'true': y_true, 'pred': p})
        for i, c in enumerate(classes): speaker_data[f'p_{c}'] = prob[:, i]
        
        speaker_preds_maj = []
        speaker_confidences = []
        speaker_preds_prob = []
        mean_probs = []
        
        for sid, s_df in speaker_data.groupby('sid'):
            # Majority Vote
            c_counter = Counter(s_df['pred'])
            winner = c_counter.most_common(1)[0][0]
            speaker_preds_maj.append((winner, s_df['true'].iloc[0]))
            speaker_confidences.append(c_counter[winner] / len(s_df))
            
            # Average Prob
            avg_p = [s_df[f'p_{cl}'].mean() for cl in classes]
            winner_p = classes[np.argmax(avg_p)]
            speaker_preds_prob.append((winner_p, s_df['true'].iloc[0]))
            mean_probs.append(np.max(avg_p))
            
        acc_maj = accuracy_score([x[1] for x in speaker_preds_maj], [x[0] for x in speaker_preds_maj]) * 100
        avg_conf = np.mean(speaker_confidences) * 100
        acc_prob = accuracy_score([x[1] for x in speaker_preds_prob], [x[0] for x in speaker_preds_prob]) * 100
        avg_prob_val = np.mean(mean_probs) * 100
        
        row[f'{m_name.upper()} Sample Acc'] = f"{s_acc:.1f}%"
        row[f'{m_name.upper()} Maj. Vote (Conf)'] = f"{acc_maj:.1f}% ({avg_conf:.1f}%)"
        row[f'{m_name.upper()} Avg. Prob (Prob)'] = f"{acc_prob:.1f}% ({avg_prob_val:.1f}%)"
        
    summary_rows.append(row)

display(pd.DataFrame(summary_rows).set_index('Data Subset'))

## 5. Per-Speaker Detail (Best Overall Model)

Colored breakdown for the model with highest overall accuracy.

In [ ]:
# Determine best model from 'All' subset sample accuracy
all_accs = {m: accuracy_score(exp_results[('All (excl. DDK)', m)][3], exp_results[('All (excl. DDK)', m)][0]) for m in MODELS}
best_m = max(all_accs, key=all_accs.get)
p, prob, classes, y_true, s_groups = exp_results[('All (excl. DDK)', best_m)]

res_df = pd.DataFrame({
    'speaker_id': gr_f, 'status': y_s_f, 'group': tg_f, 
    'true': y_s_f, 'pred': p, 'hy': y_hy[mask_no_ddk]
})
for i, c in enumerate(classes): res_df[f'prob_{c}'] = prob[:, i]

def color_f_formatted(row):
    target = row.name[1]
    ov_text = str(row['Overall Classification'])
    ov_pred = ov_text.split(' ')[0]
    gr_cols = [c for c in row.index if c in ['monologue', 'readtext', 'sentence', 'words']]
    all_groups_correct = all(row[c] == target for c in gr_cols if pd.notna(row[c]))
    styles = [''] * len(row)
    ov_pos = row.index.get_loc('Overall Classification')
    if ov_pred != target: styles[ov_pos] = 'color: red; font-weight: bold'
    elif all_groups_correct: styles[ov_pos] = 'color: green; font-weight: bold'
    for c in gr_cols: 
        if pd.notna(row[c]) and row[c] != target: styles[row.index.get_loc(c)] = 'background-color: orange'
    return styles

agg_r = []
for (sid, gid, status, hy), g_data in res_df.groupby(['speaker_id', 'group', 'status', 'hy']):
    avg_p = [g_data[f'prob_{c}'].mean() for c in classes]
    agg_r.append({'speaker_id': sid, 'group': gid, 'status': status, 'H/Y': hy, 'pred': classes[np.argmax(avg_p)]})
s_pivot = pd.DataFrame(agg_r).pivot(index=['speaker_id', 'status', 'H/Y'], columns='group', values='pred')
overall_r = []
for (sid, status), s_data in res_df.groupby(['speaker_id', 'status']):
    avg_p = [s_data[f'prob_{c}'].mean() for c in classes]
    overall_r.append({'speaker_id': sid, 'status': status, 'Overall Classification': classes[np.argmax(avg_p)], 'Overall Score': np.max(avg_p)})
ov_df = pd.DataFrame(overall_r).set_index(['speaker_id', 'status'])
final_df = ov_df.join(s_pivot.reset_index(level='H/Y'))
disp_df = final_df.copy(); disp_df['Overall Classification'] = final_df.apply(lambda x: f"{x['Overall Classification']} ({x['Overall Score']:.2f})", axis=1)
disp_df = disp_df.drop(columns=['Overall Score'])
print(f"\n--- Per-Speaker Summary ({best_m.upper()}) ---")
display(disp_df.style.apply(color_f_formatted, axis=1))